In [1]:
!git clone https://github.com/06sahar06/The-Impact-of-Weights-Quantization-in-Fake-Image-Detection.git

Cloning into 'The-Impact-of-Weights-Quantization-in-Fake-Image-Detection'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 140 (delta 6), reused 1 (delta 0), pack-reused 122 (from 1)
Receiving objects: 100% (140/140), 108.87 MiB | 33.79 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [2]:
%cd The-Impact-of-Weights-Quantization-in-Fake-Image-Detection/

/kaggle/working/The-Impact-of-Weights-Quantization-in-Fake-Image-Detection


In [15]:
!pip install -q xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 42.2 MB/s eta 0:00:0000:0100:01


In [3]:
!pip install torch torchvision diffusers transformers accelerate bitsandbytes scipy numpy Pillow opencv-python sentencepiece protobuf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 55.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 24.8 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incom

In [4]:
import os
from huggingface_hub import login, model_info
os.environ['HF_TOKEN'] = 'your huggingface token' 
login(token=os.environ['HF_TOKEN'])
for mid in ['runwayml/stable-diffusion-v1-5',
            'stabilityai/stable-diffusion-3-medium-diffusers']:
    try:
        model_info(mid)
        print(f'Access OK: {mid}')
    except Exception as e:
        print(f'FAILED: {mid}: {e}')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Access OK: runwayml/stable-diffusion-v1-5
Access OK: stabilityai/stable-diffusion-3-medium-diffusers


In [16]:
%%writefile /kaggle/working/The-Impact-of-Weights-Quantization-in-Fake-Image-Detection/img2img.py
import argparse
import os
import glob
import random
import torch
import gc
from PIL import Image

from diffusers import (
    AutoPipelineForImage2Image,
    AutoPipelineForText2Image,
    StableDiffusion3Pipeline
)

from diffusers.quantizers import PipelineQuantizationConfig

# --- Configuration & Constants ---
MODELS = {
    'sd15': 'runwayml/stable-diffusion-v1-5',
    'sd3': 'stabilityai/stable-diffusion-3-medium-diffusers',
    'sd35': 'stabilityai/stable-diffusion-3.5-medium',
}

QUANTIZATION_LEVELS = {
    'fp16': None,
    'fp8': PipelineQuantizationConfig(
        quant_backend="bitsandbytes_8bit",
        quant_kwargs={"load_in_8bit": True},
    ),
    'fp4': PipelineQuantizationConfig(
        quant_backend="bitsandbytes_4bit",
        quant_kwargs={
            "load_in_4bit": True,
            "bnb_4bit_quant_type": "nf4",
            "bnb_4bit_compute_dtype": torch.bfloat16
        }
    )
}

def flush():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

class ImageGenerator:
    def __init__(self, model_key, quantization, device=None):
        self.model_key = model_key
        self.quantization = quantization
        
        if device is None:
            if torch.cuda.is_available():
                self.device = 'cuda'
            elif torch.backends.mps.is_available():
                self.device = 'mps'
            else:
                self.device = 'cpu'
        else:
            self.device = device
            
        print(f"Using device: {self.device}")

        self.pipeline = None
        self.load_pipeline()

    def load_pipeline(self):
        print(f"Loading {self.model_key} with {self.quantization} quantization...")
        try:
            pipeline_t2i.enable_xformers_memory_efficient_attention()
        except:
            pass
        model_id = MODELS[self.model_key]
        q_config = QUANTIZATION_LEVELS[self.quantization]
        
        # Determine Dtype
        torch_dtype = None
        if self.quantization == 'fp16':
            if self.device == 'cpu':
                torch_dtype = torch.float32 
            else:
                torch_dtype = torch.float16
        else:
            torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        
        try:
            pipeline_t2i = None
                
            # --- LOGIC: SD 3/3.5 FAMILY ---
            if 'sd3' in self.model_key:
                 if self.quantization == 'fp16':
                    pipeline_t2i = StableDiffusion3Pipeline.from_pretrained(
                        model_id, torch_dtype=torch.float16
                    )
                 else:
                    pipeline_t2i = AutoPipelineForText2Image.from_pretrained(
                        model_id, quantization_config=q_config, torch_dtype=torch_dtype
                    )
                 # SD3 usually requires CPU offload to fit in consumer VRAM
                 pipeline_t2i.enable_model_cpu_offload()

            # --- LOGIC: SD1.5 ---
            else: 
                if self.quantization == 'fp16':
                    pipeline_t2i = AutoPipelineForText2Image.from_pretrained(
                        model_id, 
                        torch_dtype=torch_dtype, 
                        use_safetensors=True
                    ).to(self.device)
                else:
                    pipeline_t2i = AutoPipelineForText2Image.from_pretrained(
                        model_id, 
                        quantization_config=q_config, 
                        torch_dtype=torch_dtype,
                        use_safetensors=True
                    )
                    pipeline_t2i.enable_model_cpu_offload()

            # Convert to Image2Image pipeline
            if self.pipeline is None:
                if pipeline_t2i is not None:
                    self.pipeline = AutoPipelineForImage2Image.from_pipe(pipeline_t2i)
                    self.pipeline.enable_attention_slicing()
                    try:
                        self.pipeline.enable_vae_slicing()
                        self.pipeline.enable_vae_tiling()
                    except:
                        pass
                else:
                    raise ValueError("Failed to initialize a pipeline to convert to Img2Img.")
            
        except Exception as e:
            print(f"Error loading pipeline: {e}")
            raise e

    def generate(self, prompt, image, strength=0.3, guidance_scale=3.5, steps=30, seed=None):
        generator = None
        if seed is not None:
            generator = torch.Generator("cpu").manual_seed(seed)
        else:
             rand_seed = random.randint(0, 2**32 - 1)
             generator = torch.Generator("cpu").manual_seed(rand_seed)

        kwargs = {
            "prompt": prompt,
            "image": image,
            "strength": strength,
            "num_inference_steps": steps,
            "guidance_scale": guidance_scale,
            "generator": generator 
        }
        
        return self.pipeline(**kwargs).images[0]

def main():
    parser = argparse.ArgumentParser(description="Generate images using various Stable Diffusion models (Image-to-Image only).")
    parser.add_argument("--prompt", type=str, default="", help="Prompt for generation.")
    parser.add_argument("--models", nargs='+', default=['sd15'], choices=MODELS.keys(), help="Models to use.")
    parser.add_argument("--quantization", nargs='+', default=['fp16'], choices=QUANTIZATION_LEVELS.keys(), help="Quantization levels.")
    parser.add_argument("--input_dir", type=str, required=True, help="Directory containing input images.")
    parser.add_argument("--output_dir", type=str, default="output_img2img", help="Directory for output images.")
    parser.add_argument("--strength", type=float, default=0.3, help="Denoising strength (0.0 to 1.0).")
    parser.add_argument("--steps", type=int, default=30, help="Inference steps.")
    parser.add_argument("--guidance", type=float, default=3.5, help="Guidance scale.")
    parser.add_argument("--device", type=str, default=None, help="Device to use.")
    parser.add_argument("--seed", type=int, default=123, help="Fixed seed for reproducibility.")
    
    args = parser.parse_args()
    
    if not os.path.exists(args.input_dir):
        print(f"Input directory {args.input_dir} does not exist.")
        return

    os.makedirs(args.output_dir, exist_ok=True)
    
    input_images = sorted(glob.glob(os.path.join(args.input_dir, '*.jpg')) + 
                          glob.glob(os.path.join(args.input_dir, '*.png')) +
                          glob.glob(os.path.join(args.input_dir, '*.PNG')) +
                          glob.glob(os.path.join(args.input_dir, '*.JPG')) +
                          glob.glob(os.path.join(args.input_dir, '*.jpeg')))

    if not input_images:
        print(f"No images found in {args.input_dir}")
        return

    for model_key in args.models:
        for quant in args.quantization:
            try:
                generator = ImageGenerator(model_key, quant, device=args.device)
                
                print(f"Processing images with {model_key} ({quant})...")
                
                for i, img_path in enumerate(input_images):
                    try:
                        print(f"  Processing {os.path.basename(img_path)}...")
                        
                        input_img = Image.open(img_path).convert("RGB")
                        width, height = input_img.size

                        # Center crop 1024x1024
                        if width > 1024 and height > 1024:
                            left = (width - 1024)/2
                            top = (height - 1024)/2
                            right = (width + 1024)/2
                            bottom = (height + 1024)/2
                            input_img = input_img.crop((left, top, right, bottom))
                        
                        current_seed = args.seed + i if args.seed is not None else None
                        
                        output_img = generator.generate(
                            args.prompt, 
                            image=input_img,
                            strength=args.strength,
                            steps=args.steps, 
                            guidance_scale=args.guidance, 
                            seed=current_seed
                        )
                        
                        base_name = os.path.splitext(os.path.basename(img_path))[0]
                        seed_str = f"_seed{current_seed}" if current_seed is not None else ""
                        out_name = f"{args.output_dir}/{base_name}_{model_key}_{quant}{seed_str}.png"
                        output_img.save(out_name)
                        print(f"    Saved {out_name}")
                        
                    except Exception as e_img:
                        print(f"    Failed to process {os.path.basename(img_path)}: {e_img}")

                flush()
                del generator
                torch.cuda.empty_cache()
                flush()
                
            except Exception as e:
                print(f"Failed to load/run {model_key} with {quant}: {e}")
                flush()

if __name__ == "__main__":
    main()

Overwriting /kaggle/working/The-Impact-of-Weights-Quantization-in-Fake-Image-Detection/img2img.py


In [19]:
!python img2img.py \
  --input_dir /kaggle/input/datasets/saharguebsi/selected-imgs/selected_images \
  --prompt "oil painting, van gogh style, thick brushstrokes" \
  --models sd15\
  --quantization fp8 \
  --strength 0.6 \
  --steps 20 \
  --seed 123 \
  --output_dir dataset/fake/img2img

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Using device: cuda
Loading sd15 with fp8 quantization...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading weights:   0%|                                  | 0/196 [00:00<?, ?it/s]
Loading weights:   1%| | 1/196 [00:00<00:00, 8371.86it/s, Materializing param=te
Loading weights:   1%| | 1/196 [00:00<00:00, 2236.96it/s, Materializing param=te
Loading weights:   1%| | 2/196 [00:00<00:00, 2435.01it/s, Materializing param=te
Loading weights:   1%| | 2/196 [00:00<00:00, 1912.59it/s, M